# SNC — Modern Web App

Launches the **modern** front-end for the conditioned separator: a custom
FastAPI + JS UI with per-source control (Keep / Reduce / Remove for each
detected sound) and a public share link.

Needs a trained `separator_unet_film_multi_*.h5` on Drive (produced by
`colab_train_conditioned_separator.ipynb`). A GPU runtime makes detection
much faster but is not required.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'main'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

clone_url = (f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
             if token else
             f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('git clone failed — if the repo is private add a '
                       'GITHUB_TOKEN secret in the left sidebar.')
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Symlink saved_models to Drive so the trained checkpoints are available.
drive_models = Path(DRIVE_ROOT) / 'saved_models'
local_models = REPO_DIR / 'saved_models'
if local_models.is_symlink() or local_models.exists():
    local_models.unlink() if local_models.is_symlink() else shutil.rmtree(local_models)
local_models.symlink_to(drive_models)

found = sorted((drive_models / 'separation_models').glob('separator_unet_film_multi_*.h5'))
assert found, f'No trained model under {drive_models}/separation_models — run training first.'
print('Models found:')
for f in found:
    print('  -', f.name)

In [ ]:
# FastAPI stack; the ML deps (tensorflow/librosa) come with the Colab image.
!pip install -q fastapi 'uvicorn[standard]' python-multipart librosa==0.11.0 soundfile

## Launch the app

This cell stays running. Open the **Public URL** it prints (ends in
`.trycloudflare.com`) to use the app. Detection runs the model once per
sound class, so the first analysis on a fresh runtime can take up to a
minute; later ones are fast.

In [ ]:
!python src/application/modern_web/launch.py --share --port 8000